# ARIMA, SARIMA y SARIMAX
### El clásico que sí tienes que dominar

`Fase 3 · Video 12` — serie **Forecasting de Ventas de la A a la Z**

Con el listón puesto (Video 11), atacamos el clásico. Reutilizamos la diferenciación (Video 3), la
lectura ACF/PACF (Video 5) y las exógenas (Videos 8 y 10), y lo obligamos a superar el MASE < 1.
Spoiler: el ARIMA de libro no basta solo; ganan las **exógenas**.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))          # raíz del repo → importar src
from src.metrics import wape, mase
from src.events import get_event
from src.calendar_mx import quincena_factor_series, holiday_factor_series

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
g = df.groupby(pd.Grouper(key='date', freq='W-MON'))
y      = g['units_sold'].sum().iloc[1:-1]
promo  = g['discount'].mean().reindex(y.index)                 # intensidad promo semanal
lprice = np.log(g['unit_price'].mean()).reindex(y.index)       # log-precio semanal

H, m = 52, 52
train, test = y.iloc[:-H], y.iloc[-H:]

snaive = pd.Series(train.iloc[-m:].values[:H], index=test.index)
print(f'✅ {len(y)} semanas | horizonte de prueba: {H} | período estacional: {m}')
print(f'   Listón a superar (Seasonal Naive): WAPE {wape(test, snaive):.1%} | MASE {mase(test, snaive, train, m):.2f}')

---
## 2. El plan Box-Jenkins y el listón

ARIMA(p, d, q) junta tres piezas que ya conoces:

| Parámetro | Qué es | De dónde sale |
|---|---|---|
| **d** | Diferenciación para estacionarizar | Video 3 (ADF/KPSS) |
| **p** | Orden autorregresivo (AR) | Video 5 (PACF) |
| **q** | Orden de media móvil (MA) | Video 5 (ACF) |

Trabajamos sobre **log(demanda)** (estabiliza la varianza, Video 3) y con `d=1` (quita la tendencia). La
regla es sagrada: **cualquier modelo debe superar el benchmark** (`MASE < 1`). Si no, no vale la complejidad.

---
## 3. ARIMA no estacional: por qué no alcanza

Ajustamos un ARIMA sin componente estacional sobre `log(demanda)` y lo proyectamos un año. Fíjate en la
forma del pronóstico frente a lo real.

In [ ]:
ln = np.log1p(y)
ln_tr = ln.iloc[:-H]

arima = ARIMA(ln_tr, order=(2, 1, 2)).fit()
arima_f = pd.Series(np.expm1(arima.forecast(H).values), index=test.index)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-104:], train.values[-104:], color=BLUE, linewidth=1.1, alpha=0.6, label='train')
ax.plot(test.index, test.values, color='black', linewidth=2.5, label='REAL')
ax.plot(test.index, arima_f.values, color=RED, linewidth=1.8, label='ARIMA(2,1,2)')
ax.plot(test.index, snaive.values, color=GREEN, linewidth=1.4, alpha=0.8, label='Seasonal Naive')
ax.axvline(train.index[-1], color='black', linestyle='--', linewidth=1)
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('ARIMA no estacional: pronóstico casi plano, ciego a la estacionalidad',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.show()

print(f'ARIMA(2,1,2)   WAPE {wape(test, arima_f):.3f} | MASE {mase(test, arima_f, train, m):.2f}')
print(f'Seasonal Naive WAPE {wape(test, snaive):.3f} | MASE {mase(test, snaive, train, m):.2f}')
print('\nEl ARIMA no estacional extrapola nivel + tendencia, pero NO reproduce los picos anuales')
print('(Buen Fin, Navidad…). Su MASE ≫ 1: pierde feo contra el benchmark. Le falta la estacionalidad.')

---
## 4. SARIMA y el muro de s=52

La respuesta "de libro" es **SARIMA**: añadir un componente estacional `(P, D, Q)_s` con `s=52`. El
problema es práctico: con `s=52` statsmodels debe estimar un espacio de estados enorme con solo ~4 ciclos
de historia. En la práctica esto significa un ajuste **lento y que a menudo no converge** — y, sobre todo,
**no ofrece una vía natural para las exógenas** de calendario y promoción que necesitamos.

In [ ]:
import time
t0 = time.perf_counter()
sarima = SARIMAX(ln_tr, order=(1, 1, 1), seasonal_order=(1, 0, 0, 52),
                 enforce_stationarity=False, enforce_invertibility=False
                 ).fit(disp=False, maxiter=50)
elapsed = time.perf_counter() - t0
sarima_f = pd.Series(np.expm1(sarima.forecast(H).values), index=test.index)

print(f'SARIMA(1,1,1)(1,0,0,52)')
print(f'  tiempo de ajuste : {elapsed:.1f}s (con maxiter limitado)')
print(f'  ¿convergió?      : {sarima.mle_retvals.get("converged")}')
print(f'  WAPE {wape(test, sarima_f):.3f} | MASE {mase(test, sarima_f, train, m):.2f}')
print('\nAun cuando produce un número, el ajuste con s=52 es caro y frágil (no converge), y no le')
print('saca ventaja clara al benchmark. Peor: no hay forma limpia de sumarle las exógenas de')
print('calendario/promo. La regla práctica moderna: modelar la estacionalidad con TÉRMINOS DE')
print('FOURIER como exógenas — suave, estable y barato. Eso hacemos a continuación.')

---
## 5. SARIMAX: Fourier + exógenas de calendario y promoción

La **X** de SARIMA**X** son variables exógenas. Le damos tres bloques, todos **conocidos hacia el futuro**:

1. **Fourier** (K armónicos de período 52): la estacionalidad anual *suave*.
2. **Calendario** (eventos, quincena, festivos — Video 8): los *picos* que Fourier no capta.
3. **Promoción y precio** (Video 10): la palanca comercial.

Añadimos los bloques de forma incremental para ver cuánto aporta cada uno.

In [ ]:
def fourier_terms(index, period=52, K=6):
    t = ((index - y.index[0]).days / 7).to_numpy()
    cols = {}
    for k in range(1, K + 1):
        cols[f'sin{k}'] = np.sin(2 * np.pi * k * t / period)
        cols[f'cos{k}'] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(cols, index=index)

# Exógenas de calendario: reconstruidas desde el código del generador (Video 8),
dfull = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
evmult = np.array([(get_event(d.month, d.day).mult if get_event(d.month, d.day) else 1.0)
                   for d in dfull])
cal_daily = pd.DataFrame({
    'ev_event':    evmult,
    'ev_quincena': quincena_factor_series(dfull),
    'ev_holiday':  holiday_factor_series(dfull)[0],
}, index=dfull)
cal_week = cal_daily.resample('W-MON').mean().reindex(y.index)

F = fourier_terms(y.index)
X_fourier = F
X_cal     = pd.concat([F, cal_week], axis=1)
X_full    = pd.concat([F, cal_week, promo.rename('promo'), lprice.rename('lprice')], axis=1)

print(f'Bloques de exógenas: Fourier={F.shape[1]} cols | +calendario={X_cal.shape[1]} | +promo/precio={X_full.shape[1]}')

In [ ]:
def fit_sarimax(exog, label):
    etr, ete = exog.iloc[:-H], exog.iloc[-H:]
    mdl = SARIMAX(ln_tr, exog=etr, order=(2, 1, 1), trend='c').fit(disp=False, maxiter=200)
    fc = pd.Series(np.expm1(mdl.forecast(H, exog=ete).values), index=test.index)
    return mdl, fc, {'modelo': label, 'WAPE': wape(test, fc), 'MASE': mase(test, fc, train, m)}

m_four, f_four, r_four = fit_sarimax(X_fourier, 'SARIMAX + Fourier')
m_cal,  f_cal,  r_cal  = fit_sarimax(X_cal,     '+ calendario')
m_full, f_full, r_full = fit_sarimax(X_full,    '+ promo/precio')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test.values, color='black', linewidth=2.5, label='REAL')
ax.plot(test.index, snaive.values, color=GREEN, linewidth=1.3, alpha=0.7, label='Seasonal Naive')
ax.plot(test.index, f_four.values, color=ORANGE, linewidth=1.2, alpha=0.8, label='Fourier solo')
ax.plot(test.index, f_full.values, color=PURPLE, linewidth=2, label='SARIMAX completo')
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('SARIMAX: Fourier solo no basta; con exógenas de calendario clava los picos',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.show()

tab = pd.DataFrame([r_four, r_cal, r_full]).set_index('modelo')
print('Aporte incremental de cada bloque de exógenas:')
print(tab.assign(WAPE=lambda d: (d.WAPE*100).round(1), MASE=lambda d: d.MASE.round(2)).to_string())
print('\n→ Fourier solo es insuficiente (los picos de evento son demasiado agudos).')
print('  Las exógenas de CALENDARIO son las que hacen ganar al modelo.')

---
## 6. Diagnóstico de residuales

El ideal es **ruido blanco**: sin autocorrelación (Ljung-Box) ni estructura visible. Lo revisamos sobre el
modelo completo — y seremos honestos con lo que salga.

In [ ]:
resid = m_full.resid.iloc[m:]   # descartamos el arranque

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(resid.index, resid.values, color=PURPLE, linewidth=0.8)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Residuales del SARIMAX completo', fontsize=12, fontweight='bold')
axes[1].hist(resid.values, bins=30, color=PURPLE, edgecolor='white', alpha=0.85)
axes[1].set_title('Distribución de residuales', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

lb = acorr_ljungbox(resid, lags=[10, 20, 52], return_df=True)
print('Ljung-Box sobre residuales (H0: no hay autocorrelación):')
clean = True
for lag, row in lb.iterrows():
    ok = row['lb_pvalue'] > 0.05
    clean = clean and ok
    print(f'  lag {lag:>2}: p={row["lb_pvalue"]:.3f} → {"✅ ruido blanco" if ok else "❌ queda estructura"}')

if not clean:
    print('\nHonestidad: los residuales AÚN tienen algo de estructura (sobre todo estacional en el')
    print('lag 52): con K=6 armónicos de Fourier no capturamos del todo los picos. El modelo ya')
    print('vence al benchmark, pero es MEJORABLE — más armónicos, más exógenas o mayor orden ARIMA.')
    print('Un modelo que gana no es lo mismo que un modelo perfecto.')

---
## 7. Veredicto vs. el benchmark

¿Se ganó el SARIMAX su lugar? Comparamos todo contra el listón del Video 11.

In [ ]:
final = pd.DataFrame([
    {'modelo': 'Seasonal Naive (benchmark)', 'WAPE': wape(test, snaive),  'MASE': mase(test, snaive, train, m)},
    {'modelo': 'ARIMA(2,1,2)',               'WAPE': wape(test, arima_f), 'MASE': mase(test, arima_f, train, m)},
    {'modelo': 'SARIMAX completo',           'WAPE': wape(test, f_full),  'MASE': mase(test, f_full, train, m)},
]).set_index('modelo')

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = [GREEN if v < 1 else RED for v in final['MASE']]
ax.barh(final.index[::-1], final['MASE'][::-1], color=colors[::-1])
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='listón (seasonal naive)')
ax.set_title('MASE por modelo (< 1 le gana al benchmark)', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate(final['MASE'][::-1]):
    ax.text(v, i, f' {v:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(final.assign(WAPE=lambda d: (d.WAPE*100).round(1), MASE=lambda d: d.MASE.round(2)).to_string())
mejora = (1 - final.loc['SARIMAX completo', 'WAPE'] / final.loc['Seasonal Naive (benchmark)', 'WAPE'])
print(f'\n→ El SARIMAX completo baja el WAPE {mejora:.0%} vs el benchmark y logra MASE < 1: se ganó su lugar.')
print('  El ARIMA solo, no. La diferencia son las EXÓGENAS, no la complejidad del ARIMA.')

### Una nota sobre `auto_arima`

`auto_arima` (pmdarima) automatiza la búsqueda de (p, d, q) por AIC/BIC. Es útil, pero **no es magia**:

- **No decide tus exógenas** — y aquí fueron ellas las que ganaron el partido.
- Con `s=52` sufre el mismo muro computacional que vimos.
- No sustituye la **validación temporal**: un AIC bajo en train no garantiza buen forecast.

Úsalo para afinar el orden ARIMA *después* de haber pensado la estacionalidad, las exógenas y el esquema
de validación — no como piloto automático.

---
## 8. Conclusiones y puente al Video 13

### Las reglas que te llevas

1. **ARIMA "de libro" no basta** en una serie con estacionalidad fuerte y eventos: su pronóstico es plano.
2. **SARIMA con s=52 es impracticable.** La solución profesional: **Fourier** para la estacionalidad suave.
3. **La X de SARIMAX es la que gana el partido.** Las exógenas de calendario (Video 8) y promo/precio
   (Video 10) llevaron el modelo a **MASE < 1**.
4. **Diagnostica residuales** (Ljung-Box) y **compara siempre contra el benchmark**.
5. **`auto_arima` ayuda, no piensa por ti.**

---

### Próximo video
**Video 13 — Prophet de Meta: el puente hacia los modelos modernos**
La misma lógica de "tendencia + estacionalidad + regresores", pero con la API de Prophet — y su manejo
nativo de festivos y puntos de quiebre.